# Loading libraries

In [1]:
import pandas as pd
import numpy as np

# Loading datasets

In [59]:
## After filtering the 1000 Genomes populational dataset with bcftools to include only missense variants

pop_data_missense = pd.read_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/populational_data_missense_only.tsv",
                       sep="\t"
                       )

/var/folders/v8/pzxy2t1n2_1901rt7g0cqp_00000gn/T/ipykernel_40374/3207912597.py:3: DtypeWarning: Columns (9,10,11,12,13,56,57,103,104,105,106,107,108,109,110) have mixed types. Specify dtype option on import or set low_memory=False.
  pop_data_missense = pd.read_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/populational_data_missense_only.tsv",


# Data processing


In [60]:
## Processing the hexadecimal representations
pop_data_missense.astype(str).stack().str.contains("x3b").any()


pop_data_missense = pop_data_missense.apply(
    lambda col: col.str.replace(r"\\x3b", "|", regex=True) if col.dtype == "object" else col
)

#### **Process the 8th column**




The dataset's 8th column comprises aggregated functional annotations. The primary objective of this processing phase is to extract and transform this aggregated information into multiple distinct features.

In [61]:
pop_data_missense.rename(columns={
    "'Functional annotations: 'Allele | Annotation | Annotation_Impact | Gene_Name | Gene_ID | Feature_Type | Feature_ID | Transcript_BioType | Rank | HGVS.c | HGVS.p | cDNA.pos / cDNA.length | CDS.pos / CDS.length | AA.pos / AA.length | Distance | ERRORS / WARNINGS / INFO'":
    "vep_annotation"
    },
                         inplace=True
                         )

In [62]:
pop_data_missense_tmp = pop_data_missense.copy()
pop_data_missense_tmp["vep_annotation"] = pop_data_missense_tmp["vep_annotation"].str.split(",")

In [64]:
def align_column(col, n):
    if pd.isna(col):
        return [pd.NA] * n

    if isinstance(col, str):
        if "/" in col:
            parts = col.split("/")
            return parts[:n] + [pd.NA] * max(0, n - len(parts))
        else:
            return [col] * n

    return [col] * n



rows = []

for _, row in pop_data_missense_tmp.iterrows():
    n = len(row["vep_annotation"])

    new_row = {}
    for col in pop_data_missense_tmp.columns:
        if col == "vep_annotation":
            new_row[col] = row[col]
        else:
            new_row[col] = align_column(row[col], n)

    rows.append(pd.DataFrame(new_row))


pop_data_missense_long = (
    pd.concat(rows)
      .explode("vep_annotation")
      .reset_index(drop=True)
)

,CHROM,POS,REF,ALT,avsnp151,Func.refGene,Gene.refGene,vep_annotation,ExonicFunc.refGene,AF_EUR_unrel,...,phastCons100way_vertebrate,phastCons100way_vertebrate_rankscore,phastCons30way_mammalian,phastCons30way_mammalian_rankscore,SiPhy_29way_logOdds,SiPhy_29way_logOdds_rankscore,InterVar_automated,regsnp_fpr,regsnp_disease,regsnp_splicing_site
0,chr1,69134,A,G,rs781394307,exonic,OR4F5,G|missense_variant|MODERATE|OR4F5|ENSG00000186...,nonsynonymous_SNV,<NA>,...,0.000,0.064,0.957,0.428,8.509,0.324,Likely_benign,.,.,.
1,chr1,69149,T,A,rs1458521218,exonic,OR4F5,A|missense_variant|MODERATE|OR4F5|ENSG00000186...,nonsynonymous_SNV,<NA>,...,0.001,0.138,0.868,0.360,5.811,0.177,Likely_benign,.,.,.
2,chr1,69162,T,G,.,exonic,OR4F5,G|missense_variant|MODERATE|OR4F5|ENSG00000186...,nonsynonymous_SNV,<NA>,...,0.000,0.064,0.982,0.484,4.848,0.129,Likely_benign,.,.,.
3,chr1,69200,T,C,rs1453226491,exonic,OR4F5,C|missense_variant|MODERATE|OR4F5|ENSG00000186...,nonsynonymous_SNV,<NA>,...,0.000,0.064,0.991,0.532,5.811,0.177,Likely_benign,.,.,.
4,chr1,69224,A,T,rs568964432,exonic,OR4F5,T|missense_variant|MODERATE|OR4F5|ENSG00000186...,nonsynonymous_SNV,<NA>,...,0.004,0.166,0.575,0.285,8.509,0.324,Uncertain_significance,.,.,.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1184256,chrY,23229667,C,A,rs752477644,exonic,DAZ2|DAZ3|DAZ4,A|missense_variant&splice_region_variant|MODER...,nonsynonymous_SNV,<NA>,...,1.000,0.716,0.969,0.448,.,.,Uncertain_significance,.,.,.
1184257,chrY,24362373,G,C,.,intergenic,GOLGA2P2Y|TTTY17A,C|missense_variant|MODERATE|AC012005.1|ENSG000...,.,<NA>,...,.,.,.,.,.,.,.,.,.,.
1184258,chrY,24362375,C,G,.,intergenic,GOLGA2P2Y|TTTY17A,G|missense_variant|MODERATE|AC012005.1|ENSG000...,.,<NA>,...,.,.,.,.,.,.,.,.,.,.
1184259,chrY,24864466,T,A,.,exonic,DAZ2|DAZ3|DAZ4,A|missense_variant|MODERATE|DAZ4|ENSG000002059...,nonsynonymous_SNV,<NA>,...,1.000,0.716,0.569,0.284,.,.,Uncertain_significance,.,.,.


In [66]:
new_headers = [
    "Allele", "Annotation", "Annotation_Impact", "Gene_Name",
    "Gene_ID", "Feature_Type", "Feature_ID", "Transcript_BioType",
    "Rank", "HGVS_c", "HGVS_p", "cDNA_pos_len",
    "CDS_pos_len", "AA_pos_len", "Distance", "ERRORS_WARNINGS_INFO"
]

target_col = "vep_annotation"

split_df = pop_data_missense_long[target_col].str.split('|', expand=True)

split_df = split_df.iloc[:, :len(new_headers)]
split_df.columns = new_headers

col_index = pop_data_missense_long.columns.get_loc(target_col)

# The "slice and stitch" strategy
# Part A: columns from start up to (but not including) the target
part_a = pop_data_missense_long.iloc[:, :col_index]

# Part B: the new split columns (split_df)

# Part C: columns from immediately after the target to the end
part_c = pop_data_missense_long.iloc[:, col_index+1:]

# Combining them
pop_data_missense_long_split = pd.concat([part_a, split_df, part_c], axis=1)

pop_data_missense_long_split.head()

,CHROM,POS,REF,ALT,avsnp151,Func.refGene,Gene.refGene,Allele,Annotation,Annotation_Impact,...,phastCons100way_vertebrate,phastCons100way_vertebrate_rankscore,phastCons30way_mammalian,phastCons30way_mammalian_rankscore,SiPhy_29way_logOdds,SiPhy_29way_logOdds_rankscore,InterVar_automated,regsnp_fpr,regsnp_disease,regsnp_splicing_site
0,chr1,69134,A,G,rs781394307,exonic,OR4F5,G,missense_variant,MODERATE,...,0.000,0.064,0.957,0.428,8.509,0.324,Likely_benign,.,.,.
1,chr1,69149,T,A,rs1458521218,exonic,OR4F5,A,missense_variant,MODERATE,...,0.001,0.138,0.868,0.360,5.811,0.177,Likely_benign,.,.,.
2,chr1,69162,T,G,.,exonic,OR4F5,G,missense_variant,MODERATE,...,0.000,0.064,0.982,0.484,4.848,0.129,Likely_benign,.,.,.
3,chr1,69200,T,C,rs1453226491,exonic,OR4F5,C,missense_variant,MODERATE,...,0.000,0.064,0.991,0.532,5.811,0.177,Likely_benign,.,.,.
4,chr1,69224,A,T,rs568964432,exonic,OR4F5,T,missense_variant,MODERATE,...,0.004,0.166,0.575,0.285,8.509,0.324,Uncertain_significance,.,.,.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1184256,chrY,23229667,C,A,rs752477644,exonic,DAZ2|DAZ3|DAZ4,A,missense_variant&splice_region_variant,MODERATE,...,1.000,0.716,0.969,0.448,.,.,Uncertain_significance,.,.,.
1184257,chrY,24362373,G,C,.,intergenic,GOLGA2P2Y|TTTY17A,C,missense_variant,MODERATE,...,.,.,.,.,.,.,.,.,.,.
1184258,chrY,24362375,C,G,.,intergenic,GOLGA2P2Y|TTTY17A,G,missense_variant,MODERATE,...,.,.,.,.,.,.,.,.,.,.
1184259,chrY,24864466,T,A,.,exonic,DAZ2|DAZ3|DAZ4,A,missense_variant,MODERATE,...,1.000,0.716,0.569,0.284,.,.,Uncertain_significance,.,.,.


In [67]:
pop_data_missense_long_split.insert(
    0,
    "var_id",
    pop_data_missense_long_split[["CHROM","POS","REF","ALT"]].astype(str).agg(":".join, axis=1)
)


pop_data_missense_long_split["var_id"] = pop_data_missense_long_split["var_id"].str.replace("chr", "", regex=False)

In [68]:
pop_data_missense_long_split[["AA_pos", "AA_len"]] = pop_data_missense_long_split["AA_pos_len"].str.split("/", expand=True)
pop_data_missense_long_split = pop_data_missense_long_split.drop(columns="AA_len")

In [69]:
pop_data_missense_long_split = pop_data_missense_long_split.replace('.', np.nan)

In [70]:
# Edit column "HGVS_p"
target_col = 'HGVS_p'

pattern = r'p\.([A-Z][a-z]+)\d+([A-Z][a-z]+)'
new_cols = pop_data_missense_long_split[target_col].str.extract(pattern)
new_cols.columns = ['Ref_AA', 'Alt_AA']  # Name the new columns

col_index = pop_data_missense_long_split.columns.get_loc(target_col)

# Part A: everything before HGVS_p
part_a = pop_data_missense_long_split.iloc[:, :col_index]

# Part B: the new columns (new_cols)

# Part C: everything after HGVS_p (using index+1 to skip the old column)
part_c = pop_data_missense_long_split.iloc[:, col_index+1:]

# 5. Concatenate them together
pop_data_missense_long_split = pd.concat([part_a, new_cols, part_c], axis=1)

# Result
pop_data_missense_long_split[['Ref_AA', 'Alt_AA']]

,Ref_AA,Alt_AA
0,Glu,Gly
1,Leu,Gln
2,Phe,Leu
3,Leu,Pro
4,Asp,Val
...,...,...
1184256,Gln,Lys
1184257,Leu,Val
1184258,Arg,Thr
1184259,Ser,Arg


!!!! COlocar no novo notebook do Anovar

In [ ]:
acmg_map = {'Benign': 0, 'Likely_benign': 1, 'Uncertain_significance': 2, 'Likely_pathogenic': 3, 'Pathogenic': 4}
polyphen_map = {'D': 1.0, 'P': 0.5, 'B': 0.0}


pop_data_missense_long_split['Polyphen2_HVAR_pred'] = pop_data_missense_long_split['Polyphen2_HVAR_pred'].map(polyphen_map)
pop_data_missense_long_split['Polyphen2_HDIV_pred'] = pop_data_missense_long_split['Polyphen2_HDIV_pred'].map(polyphen_map)
pop_data_missense_long_split['InterVar_automated'] = pop_data_missense_long_split['InterVar_automated'].map(acmg_map)

In [13]:
pop_data_missense_long_split = pop_data_missense_long_split[~pop_data_missense_long_split['Ref_AA'].isna()]

#### **Subset by the Annovar annotation**

In [71]:
pop_data_missense_long_split.columns.to_list()

['var_id',
 'CHROM',
 'POS',
 'REF',
 'ALT',
 'avsnp151',
 'Func.refGene',
 'Gene.refGene',
 'Allele',
 'Annotation',
 'Annotation_Impact',
 'Gene_Name',
 'Gene_ID',
 'Feature_Type',
 'Feature_ID',
 'Transcript_BioType',
 'Rank',
 'HGVS_c',
 'Ref_AA',
 'Alt_AA',
 'cDNA_pos_len',
 'CDS_pos_len',
 'AA_pos_len',
 'Distance',
 'ERRORS_WARNINGS_INFO',
 'ExonicFunc.refGene',
 'AF_EUR_unrel',
 'AF_EAS_unrel',
 'AF_AMR_unrel',
 'AF_SAS_unrel',
 'AF_AFR_unrel',
 'ACMG_GENE',
 'ACMG_PATH',
 'dbSNPBuildID',
 'Polyphen2_HVAR_pred',
 'Polyphen2_HDIV_pred',
 'Uniprot_aapos_Polyphen2',
 'CADD_phred',
 'LoFtool_score',
 'RVIS_percentile',
 'RVIS',
 'GDI',
 'GDI_Phred',
 'Entrez_gene_id',
 'Pathway_BioCarta_short',
 'CLNSIG',
 'SIFT_score',
 'SIFT_converted_rankscore',
 'SIFT_pred',
 'SIFT4G_score',
 'SIFT4G_converted_rankscore',
 'SIFT4G_pred',
 'LRT_score',
 'LRT_converted_rankscore',
 'LRT_pred',
 'MutationTaster_score',
 'MutationTaster_converted_rankscore',
 'MutationTaster_pred',
 'MutationAssess

In [ ]:
annovar_columns = pop_data_missense_long_split.iloc[:, 33:134].columns.to_list()
pop_data_missense_annovar = pop_data_missense_long_split[['var_id'] + annovar_columns]
pop_data_missense_long_split = pop_data_missense_long_split.drop(columns=annovar_columns)

In [76]:
pop_data_missense_annovar.drop_duplicates(inplace=True)
pop_data_missense_annovar.reset_index(drop=True, inplace=True)

pop_data_missense_long_split.drop_duplicates(inplace=True)
pop_data_missense_long_split.reset_index(drop=True, inplace=True)

,var_id,dbSNPBuildID,Polyphen2_HVAR_pred,Polyphen2_HDIV_pred,Uniprot_aapos_Polyphen2,CADD_phred,LoFtool_score,RVIS_percentile,RVIS,GDI,...,phastCons100way_vertebrate,phastCons100way_vertebrate_rankscore,phastCons30way_mammalian,phastCons30way_mammalian_rankscore,SiPhy_29way_logOdds,SiPhy_29way_logOdds_rankscore,InterVar_automated,regsnp_fpr,regsnp_disease,regsnp_splicing_site
0,1:69134:A:G,144,B,B,15,10.15,NaN,NaN,NaN,113.16669,...,0.000,0.064,0.957,0.428,8.509,0.324,Likely_benign,NaN,NaN,NaN
1,1:69149:T:A,NaN,D,D,20,23.0,NaN,NaN,NaN,113.16669,...,0.001,0.138,0.868,0.360,5.811,0.177,Likely_benign,NaN,NaN,NaN
2,1:69162:T:G,NaN,P,P,24,11.95,NaN,NaN,NaN,113.16669,...,0.000,0.064,0.982,0.484,4.848,0.129,Likely_benign,NaN,NaN,NaN
3,1:69200:T:C,NaN,P,P,37,13.03,NaN,NaN,NaN,113.16669,...,0.000,0.064,0.991,0.532,5.811,0.177,Likely_benign,NaN,NaN,NaN
4,1:69224:A:T,142,D,D,45,23.2,NaN,NaN,NaN,113.16669,...,0.004,0.166,0.575,0.285,8.509,0.324,Uncertain_significance,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
910032,Y:23229667:C:A,144,D,P,190,24.5,NaN,NaN,NaN,NaN,...,1.000,0.716,0.969,0.448,NaN,NaN,Uncertain_significance,NaN,NaN,NaN
910033,Y:24362373:G:C,NaN,NaN,NaN,NaN,6.758,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
910034,Y:24362375:C:G,NaN,NaN,NaN,NaN,9.428,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
910035,Y:24864466:T:A,NaN,P,P,"220,220,269,269",20.1,NaN,NaN,NaN,NaN,...,1.000,0.716,0.569,0.284,NaN,NaN,Uncertain_significance,NaN,NaN,NaN


#### **Save datasets**

In [77]:
pop_data_missense_long_split.to_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/Populational_Data/pop_data_missense_final.csv",
                                    sep="\t",
                                    index=False,
                                    na_rep="NA")

pop_data_missense_annovar.to_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/Populational_Data/pop_data_missense_annovar.csv",
                                    sep="\t",
                                    index=False,
                                    na_rep="NA")